In [6]:
#!git clone https://github.com/alexeygrigorev/clothing-dataset.git

In [ ]:
%load_ext autoreload
%autoreload 2

from run_epoch import *
from profiler import Profile, Schedule

from torch.profiler import schedule
from torch.profiler import profile, ProfilerActivity, record_function

Unknown instance spec: Please select VM configuration

# torch profiler

In [ ]:
def run_epoch(model, train_loader, val_loader, criterion, optimizer, steps = 10) -> tp.Tuple[float, float]:
    epoch_loss, epoch_accuracy = 0, 0
    val_loss, val_accuracy = 0, 0
    model.train()
    train_iter = iter(train_loader)
    for _ in tqdm(range(steps), desc="Train"):
        data, label = next(train_iter)
        data = data.to(Settings.device)
        label = label.to(Settings.device)
        output = model(data)
        loss = criterion(output, label)
        acc = (output.argmax(dim=1) == label).float().mean()
        epoch_accuracy += acc.item() / len(train_loader)
        epoch_loss += loss.item() / len(train_loader)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    model.eval()
    val_iter = iter(val_loader)
    for _ in tqdm(range(steps), desc="val"):
        data, label = next(val_iter)
        data = data.to(Settings.device)
        label = label.to(Settings.device)
        output = model(data)
        loss = criterion(output, label)
        acc = (output.argmax(dim=1) == label).float().mean()
        val_accuracy += acc.item() / len(train_loader)
        val_loss += loss.item() / len(train_loader)

    return epoch_loss, epoch_accuracy, val_loss, val_accuracy

In [ ]:
seed_everything()
model = get_vit_model()
train_loader, val_loader = get_loaders()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=Settings.lr)

activities = [ProfilerActivity.CUDA, ProfilerActivity.CPU]
epochs = 5
schedule = torch.profiler.schedule(warmup=1, active=epochs-1)
with profile(activities=activites, ) as p:
    for _ in range(epochs):
        run_epoch(model, train_loader, val_loader, criterion, optimizer)
        p.step()
    p.export_chrome_trace("my_trace.json")

# my profiler

In [27]:
def run_epoch(model, train_loader, val_loader, criterion, optimizer, prof_schedule : Schedule) -> tp.Tuple[float, float]:
    epoch_loss, epoch_accuracy = 0, 0
    val_loss, val_accuracy = 0, 0
    model.train()
    with Profile(model, prof_schedule) as p:
        data_iter = iter(train_loader)
        for _ in tqdm(range(10), desc="Train"):
            data, label = next(data_iter)
            data = data.to(Settings.device)
            label = label.to(Settings.device)
            output = model(data)
            loss = criterion(output, label)
            acc = (output.argmax(dim=1) == label).float().mean()
            epoch_accuracy += acc.item() / len(train_loader)
            epoch_loss += loss.item() / len(train_loader)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            p.step()
        #p.summary()
        p.to_perfetto()
    return epoch_loss, epoch_accuracy, val_loss, val_accuracy

seed_everything()
model = get_vit_model()
train_loader, val_loader = get_loaders()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=Settings.lr)

schedule = Schedule(1, 1, 1, 5, 2)

run_epoch(model, train_loader, val_loader, criterion, optimizer, schedule)

Dataset already extracted
Train Data: 4322
Val Data: 1081


AttributeError: type object 'Schedule' has no attribute 'Skip'

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 23.0.1 -> 26.1.2
[notice] To update, run: python3 -m pip install --upgrade pip


In [4]:
pth = "/home/jupyter/project/efdl/ysda_efdl_speedrun/LossScaling_Batching_Profiling/task3/clothing-dataset/images"

from pathlib import Path

# Replace with your directory path
dir_path = Path(pth)

# Count only files in the immediate directory
file_count = sum(1 for item in dir_path.iterdir() if item.is_file())

print(f"Number of files: {file_count}")

Number of files: 5756
